# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Memory Monitoring Utilities

In [3]:
def get_memory_usage():
    """Get current memory usage in MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def format_bytes(bytes):
    """Format bytes to human readable format"""
    for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
        if bytes < 1024.0:
            return f"{bytes:.2f} {unit}"
        bytes /= 1024.0
    return f"{bytes:.2f} PB"

def estimate_chunk_memory(width, height, bands, dtype):
    """Estimate memory requirement for a chunk"""
    dtype_sizes = {
        'uint8': 1, 'uint16': 2, 'uint32': 4,
        'int8': 1, 'int16': 2, 'int32': 4,
        'float32': 4, 'float64': 8
    }
    bytes_per_pixel = dtype_sizes.get(str(dtype), 4)
    return width * height * bands * bytes_per_pixel

def calculate_optimal_chunk_size(raster_width, raster_height, bands, dtype, memory_limit_mb=500):
    """Calculate optimal chunk size based on available memory"""
    memory_limit_bytes = memory_limit_mb * 1024 * 1024
    
    # Start with default chunk size
    chunk_size = 1024
    
    # Calculate memory for default chunk
    chunk_memory = estimate_chunk_memory(chunk_size, chunk_size, bands, dtype)
    
    # Adjust chunk size if needed
    if chunk_memory > memory_limit_bytes:
        # Calculate maximum chunk size that fits in memory
        bytes_per_pixel = chunk_memory / (chunk_size * chunk_size)
        max_pixels = memory_limit_bytes / bytes_per_pixel
        chunk_size = int(np.sqrt(max_pixels))
        # Round down to nearest power of 2 for efficiency
        chunk_size = 2 ** int(np.log2(chunk_size))
    
    # Ensure chunk size is at least 256
    chunk_size = max(256, chunk_size)
    
    print(f"📊 Optimal chunk size: {chunk_size}x{chunk_size}")
    print(f"   Estimated memory per chunk: {format_bytes(estimate_chunk_memory(chunk_size, chunk_size, bands, dtype))}")
    
    return chunk_size

print("✅ Memory monitoring utilities loaded")

✅ Memory monitoring utilities loaded


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [4]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [5]:
EVENT_NAME = '202412_Earthquake_CA'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'opera'      #find the name within drcs_activations OLD Directory (see link above)

RENAME_PRODUCT = 'Sentinel-1'   #choose from LIST of 2nd level directories (see above list)

PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory
DIRECTORY_NEW = f'{DIR_NEW_BASE}/{RENAME_PRODUCT}'

In [6]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = {
    "driver": "COG",
    "compress": "DEFLATE",
}

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [7]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 4 .tif files in the S3 bucket.


['drcs_activations/202412_Earthquake_CA/opera/displacement/NorCal_S1_20241125_20241207_AZ.tif',
 'drcs_activations/202412_Earthquake_CA/opera/displacement/NorCal_S1_20241125_20241207_UNW.tif',
 'drcs_activations/202412_Earthquake_CA/opera/displacement/NorCal_S1_20241125_20241207_WRP.tif',
 'drcs_activations/202412_Earthquake_CA/opera/displacement/NorCal_S1_20241207_20241125_RNG.tif']

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


In [13]:
config_aria = {
    "data_acquisition_method": "s3",
    "raw_data_bucket" : BUCKET, #DO NOT CHANGE
    "raw_data_prefix": PATH_OLD,
    "cog_data_bucket": BUCKET, #DO NOT CHANGE
    "cog_data_prefix": f"{DIRECTORY_NEW}/aria",  #We changed this!!!!!!!
    "local_output_dir": f"output/{EVENT_NAME}",  # Local directory to save COGs
    "transformation": {}
}

## Configure bucket and paths (no need to create session manually)

In [14]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

In [15]:
# Use the function with config_WM
cir_bucket = return_bucket_info(config_aria)


Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202501_Fire_CA/aria
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/HLS/aria


## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [16]:
def makedirs(name):
    # Create necessary directories
    os.makedirs("reproj", exist_ok=True)
    
    # Create data_download directory for caching
    data_download_dir = "data_download"
    os.makedirs(data_download_dir, exist_ok=True)
    
    # Create subdirectory structure to match S3 path
    s3_path_parts = name.split('/')
    local_subdir = os.path.join(data_download_dir, *s3_path_parts[:-1])
    os.makedirs(local_subdir, exist_ok=True)

    # Local path for the downloaded file (persistent storage)
    local_download_path = os.path.join(data_download_dir, name)
    
    return data_download_dir, local_subdir, local_download_path

In [17]:
def convert_to_proper_CRS_and_cogify_chunked(name, cog_filename, cog_data_bucket, cog_data_prefix, 
                                            local_output_dir=None, chunk_config=None):
    """
    Convert a file to Cloud Optimized GeoTIFF with proper CRS using chunked processing.
    
    This function includes:
    - Chunked processing for memory efficiency
    - Download caching to avoid re-downloading files
    - CRS reprojection to EPSG:4326
    - COG validation before upload
    - Upload to S3
    - Smart nodata value handling based on data type
    - Memory monitoring and progress tracking
    """
    if chunk_config is None:
        chunk_config = CHUNK_CONFIG
    
    s3_key = f"{cog_data_prefix}/{cog_filename}"
    reproject_filename = f"reproj/{cog_filename}"

    #Make directories
    data_download_dir, local_subdir, local_download_path = makedirs(name)
    
    # Temporary file for processing
    temp_input_file = f"temp_{os.path.basename(name)}"
    
    # Memory monitoring
    if chunk_config.get('enable_memory_monitoring', True):
        initial_memory = get_memory_usage()
        print(f"   [MEMORY] Initial: {initial_memory:.1f} MB")

    try:
        import shutil
        
        # Check if file already exists locally
        if os.path.exists(local_download_path):
            print(f"   [CACHE HIT] Using cached file: {local_download_path}")
            shutil.copy(local_download_path, temp_input_file)
        else:
            # Download the file from S3
            print(f"   [DOWNLOAD] Downloading from S3...")
            s3_client.download_file(BUCKET, name, local_download_path)
            print(f"   [DOWNLOAD] ✅ Saved to cache")
            shutil.copy(local_download_path, temp_input_file)
        
        # Open source file and get metadata
        with rasterio.open(temp_input_file) as src:
            dst_crs = "EPSG:4326"
            
            # Check if reprojection is needed
            if src.crs and src.crs.to_string() == dst_crs:
                print(f"   [REPROJECT] Already in {dst_crs}, skipping reprojection")
                import shutil
                shutil.copy(temp_input_file, reproject_filename)
            else:
                print(f"   [REPROJECT] Converting to EPSG:4326 using chunked processing...")
                
                # Calculate transform for destination
                transform, width, height = calculate_default_transform(
                    src.crs, dst_crs, src.width, src.height, *src.bounds
                )
                
                # Calculate optimal chunk size
                chunk_size = calculate_optimal_chunk_size(
                    width, height, src.count, src.dtypes[0],
                    memory_limit_mb=chunk_config.get('memory_limit_mb', 500)
                )
                
                # Prepare output kwargs
                kwargs = src.meta.copy()
                kwargs.update({
                    "driver": "GTiff",  # Use GTiff for intermediate file
                    "compress": "DEFLATE",
                    "crs": dst_crs,
                    "transform": transform,
                    "width": width,
                    "height": height,
                    "tiled": True,
                    "blockxsize": min(chunk_size, width),
                    "blockysize": min(chunk_size, height)
                })
                
                # Create output file
                with rasterio.open(reproject_filename, "w", **kwargs) as dst:
                    # Calculate number of chunks
                    n_chunks_x = (width + chunk_size - 1) // chunk_size
                    n_chunks_y = (height + chunk_size - 1) // chunk_size
                    total_chunks = n_chunks_x * n_chunks_y
                    
                    print(f"   [CHUNKS] Processing {total_chunks} chunks ({n_chunks_x}x{n_chunks_y})")
                    
                    # Process each band
                    for band_idx in range(1, src.count + 1):
                        print(f"   [BAND {band_idx}/{src.count}] Processing...")
                        
                        # Use tqdm for progress tracking if enabled
                        if chunk_config.get('show_progress', True):
                            chunk_iterator = tqdm(
                                total=total_chunks,
                                desc=f"Band {band_idx}",
                                unit="chunks",
                                leave=False
                            )
                        else:
                            chunk_iterator = None
                        
                        # Process chunks
                        for y in range(0, height, chunk_size):
                            for x in range(0, width, chunk_size):
                                # Define window for this chunk
                                win_width = min(chunk_size, width - x)
                                win_height = min(chunk_size, height - y)
                                window = Window(x, y, win_width, win_height)
                                
                                # Create temporary arrays for chunk
                                chunk_data = np.zeros((win_height, win_width), dtype=src.dtypes[0])
                                
                                # Reproject chunk
                                reproject(
                                    source=rasterio.band(src, band_idx),
                                    destination=chunk_data,
                                    src_transform=src.transform,
                                    src_crs=src.crs,
                                    dst_transform=transform * rasterio.windows.transform(window, transform),
                                    dst_crs=dst_crs,
                                    resampling=Resampling.nearest,
                                    wrapdateline=True
                                )
                                
                                # Write chunk to output
                                dst.write(chunk_data, band_idx, window=window)
                                
                                # Update progress
                                if chunk_iterator:
                                    chunk_iterator.update(1)
                                
                                # Force garbage collection periodically
                                if (y // chunk_size * n_chunks_x + x // chunk_size) % 10 == 0:
                                    gc.collect()
                                    
                                    if chunk_config.get('enable_memory_monitoring', True):
                                        current_memory = get_memory_usage()
                                        if current_memory > initial_memory * 2:
                                            print(f"\n   [MEMORY] High usage: {current_memory:.1f} MB, forcing cleanup...")
                                            gc.collect()
                        
                        if chunk_iterator:
                            chunk_iterator.close()
        
        # COGify & upload
        print(f"   [COGIFY] Creating COG from reprojected file...")
        
        # Use rasterio to create COG
        with rasterio.open(reproject_filename) as src:
            # Smart nodata value handling based on data type
            print(f"   [NODATA] Data type: {src.dtypes[0]}")
            if src.dtypes[0] == 'uint8':
                nodata_value = 0
                print(f"   [NODATA] Using nodata value {nodata_value} for uint8 data")
            elif src.dtypes[0] == 'uint16':
                nodata_value = 0
                print(f"   [NODATA] Using nodata value {nodata_value} for uint16 data")
            else:
                nodata_value = -9999
                print(f"   [NODATA] Using nodata value {nodata_value} for {src.dtypes[0]} data")
            
            # Update profile for COG
            profile = src.profile.copy()
            profile.update(COG_PROFILE)
            profile['nodata'] = nodata_value
            
            with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
                tmp_name = tmp.name
                
                # Write COG using chunked approach
                with rasterio.open(tmp_name, 'w', **profile) as dst:
                    # Process in chunks to avoid memory issues
                    for band_idx in range(1, src.count + 1):
                        for y in range(0, src.height, chunk_size):
                            for x in range(0, src.width, chunk_size):
                                win_width = min(chunk_size, src.width - x)
                                win_height = min(chunk_size, src.height - y)
                                window = Window(x, y, win_width, win_height)
                                
                                data = src.read(band_idx, window=window)
                                dst.write(data, band_idx, window=window)
                
                # Validate COG
                print(f"   [VALIDATE] Checking COG validity...")
                is_valid_cog, validation_details = validate_cog(tmp_name)
                
                if is_valid_cog:
                    print(f"   [VALIDATE] ✅ Valid COG")
                else:
                    print(f"   [VALIDATE] ⚠️ COG validation warnings")
                    critical_errors = [e for e in validation_details.get('errors', []) if 'Invalid driver' in e]
                    if critical_errors:
                        raise ValueError(f"Critical COG validation failed")
                    if 'errors' in validation_details:
                        for error in validation_details['errors']:
                            print(f"      - {error}")
                    if 'warnings' in validation_details:
                        for warning in validation_details['warnings']:
                            print(f"      - {warning}")
                
                # Upload to S3
                print(f"   [UPLOAD] Uploading to S3...")
                s3_client.upload_file(
                    Filename=tmp_name,
                    Bucket=cog_data_bucket,
                    Key=s3_key
                )
                print(f"   [SUCCESS] ✅ Uploaded to s3://{cog_data_bucket}/{s3_key}")
                
                # Save locally if specified
                if local_output_dir:
                    os.makedirs(local_output_dir, exist_ok=True)
                    local_path = os.path.join(local_output_dir, cog_filename)
                    import shutil
                    shutil.copy(tmp_name, local_path)
        
        # Final memory report
        if chunk_config.get('enable_memory_monitoring', True):
            final_memory = get_memory_usage()
            print(f"   [MEMORY] Final: {final_memory:.1f} MB (Change: {final_memory - initial_memory:+.1f} MB)")
            
    except Exception as e:
        print(f"   [ERROR] Failed: {str(e)}")
        raise
            
    finally:
        # Clean up temporary files
        for temp_file in [temp_input_file, reproject_filename]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        if 'tmp_name' in locals() and os.path.exists(tmp_name):
            os.remove(tmp_name)
        
        # Force final garbage collection
        gc.collect()

print("✅ Chunked COG conversion function defined with memory-efficient processing")

✅ Chunked COG conversion function defined with memory-efficient processing


In [18]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 7
  - Total size: 0.01 GB

📁 Cached files (first 10):
  - drcs_activations/202506_Fire_NM/aria/dist_hls/OPERA-DIST-ALERT-HLS-DIST-STATUS_20250624.tif (0.4 MB)
  - drcs_activations/202506_Fire_NM/aria/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250624.tif (0.5 MB)
  - drcs_activations/202506_Fire_NM/aria/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250626.tif (0.5 MB)
  - drcs_activations/202506_Fire_NM/aria/dist_hls/OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250626.tif (0.4 MB)
  - drcs_activations/202506_Fire_NM/aria/dswx_hls/OPERA_DSWx-HLS_V1_BWTR_2025-06-23.tif (1.2 MB)
  - drcs_activations/202506_Fire_NM/aria/dswx_hls/OPERA_DSWx-HLS_V1_BWTR_2025-06-26.tif (1.3 MB)
  - drcs_activations/202506_Fire_NM/aria/dswx_hls/OPERA_DSWx-HLS_V1_BWTR_2025-06-28.tif (1.0 MB)


(7, 5572085)

# Process files

In [35]:
f='drcs_activations/202501_Fire_CA/aria/dist/dist_s1_prototype/disturbance_track64_2025-01-09_share.tif'


disturbance_track64_2025-01-09_share
['disturbance', 'track64', '2025-01-09', 'share']
202501_Fire_CA_disturbance_track64_share_2025-01-09day.tif


In [28]:
# Define filename creator functions for different file types

def create_cog_filename_k1(f, EVENT_NAME):
    """Create COG filename for water mask files."""
    f2 = Path(f).stem
    print(f2)
    
    f3 = f.split('/')
    # print(f3)
    f4 = f3[1].split('_')[0]
    # print(f4)
    cog_filename= f'{EVENT_NAME}_{f2}_{f4}month.tif'
    # print(cog_filename)

    return cog_filename


# Test functions
print("Testing k1 filename:")
test_k1 = create_cog_filename_k1('drcs_activations/202501_Fire_CA/aria/asf/Eaton.tif', EVENT_NAME)
print(f"  {test_k1}")

Testing k1 filename:
Eaton
  202501_Fire_CA_Eaton_202501month.tif


In [30]:
def create_cog_filename_k2(f, EVENT_NAME):
    """Create COG filename for water mask files."""
    f2 = Path(f).stem
    f2
    cog_filename= f'{EVENT_NAME}_{f2}day.tif'

    return cog_filename
    
# Test functions
print("Testing k2 filename:")
test_k2 = create_cog_filename_k2('drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250112.tif', EVENT_NAME)
print(f"  {test_k2}")

Testing k2 filename:
  202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250112day.tif


In [37]:
def create_cog_filename_k3(f, EVENT_NAME):
    """Create COG filename for water mask files."""
    f2 = Path(f).stem
    # print(f2)
    
    f2split = f2.split('_')
    # print(f2split)
    
    cog_filename= f'{EVENT_NAME}_{f2split[0]}_{f2split[1]}_{f2split[3]}_{f2split[2]}day.tif'
    
    # print(cog_filename)
    return cog_filename
    
# Test functions
print("Testing k2 filename:")
test_k3 = create_cog_filename_k3('drcs_activations/202501_Fire_CA/aria/dist/dist_s1_prototype/disturbance_track64_2025-01-09_share.tif', EVENT_NAME)
print(f"  {test_k3}")

Testing k2 filename:
  202501_Fire_CA_disturbance_track64_share_2025-01-09day.tif


In [38]:
# Process cir files with chunked processing
if k1:
    print("\n" + "="*50)
    print("🌊 Processing Water Mask Files (Chunked)")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    # Use the chunked conversion function instead of the original
    def chunked_converter(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, cog_filename, cog_data_bucket, cog_data_prefix, 
            local_output_dir, chunk_config=CHUNK_CONFIG
        )
    
    wm_results = process_file_batch(
        file_list=k1,
        s3_client=s3_client,
        config=cir_bucket,
        filename_creator_func=create_cog_filename_k1,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, wm_results], ignore_index=True)
    
    # Print overall summary
    print_batch_summary(all_files_processed)


🌊 Processing Water Mask Files (Chunked)
Eaton

[1/2] Processing: drcs_activations/202501_Fire_CA/aria/asf/Eaton.tif
   Output filename: 202501_Fire_CA_Eaton_202501month.tif
   [MEMORY] Initial: 296.0 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [CHUNKS] Processing 12 chunks (4x3)
   [BAND 1/1] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_Eaton_202501month.tif
   [MEMORY] Final: 332.4 MB (Change: +36.4 MB)
   ✅ Generated and saved COG: 202501_Fire_CA_Eaton_202501month.tif
Palisades

[2/2] Processing: drcs_activations/202501_Fire_CA/aria/asf/Palisades.tif
   Output filename: 202501_Fire_CA_Palisades_202501month.tif
   [MEMORY] Initial: 332.4 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [CHUNKS] Processing 12 chunks (4x3)
   [BAND 1/1] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_Palisades_202501month.tif
   [MEMORY] Final: 339.4 MB (Change: +6.9 MB)
   ✅ Generated and saved COG: 202501_Fire_CA_Palisades_202501month.tif

✅ Batch processing complete: 2 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/aria/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/aria/files_converted.csv

📊 BATCH PROCESSING SUMMARY
Total files processed: 2
Successful: 2
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-02T20:36:12.118859


In [39]:
# Process cir files with chunked processing
if k2:
    print("\n" + "="*50)
    print("🌊 Processing Water Mask Files (Chunked)")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    # Use the chunked conversion function instead of the original
    def chunked_converter(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, cog_filename, cog_data_bucket, cog_data_prefix, 
            local_output_dir, chunk_config=CHUNK_CONFIG
        )
    
    wm_results = process_file_batch(
        file_list=k2,
        s3_client=s3_client,
        config=cir_bucket,
        filename_creator_func=create_cog_filename_k2,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, wm_results], ignore_index=True)
    
    # Print overall summary
    print_batch_summary(all_files_processed)


🌊 Processing Water Mask Files (Chunked)

[1/4] Processing: drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250112.tif
   Output filename: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250112day.tif
   [MEMORY] Initial: 339.4 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 4.00 MB
   [CHUNKS] Processing 16 chunks (4x4)
   [BAND 1/1] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250112day.tif
   [MEMORY] Final: 427.4 MB (Change: +88.1 MB)
   ✅ Generated and saved COG: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250112day.tif

[2/4] Processing: drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250114.tif
   Output filename: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250114day.tif
   [MEMORY] Initial: 427.4 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated m

   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250114day.tif
   [MEMORY] Final: 427.5 MB (Change: +0.0 MB)
   ✅ Generated and saved COG: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-ANOM-MAX_20250114day.tif

[3/4] Processing: drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250112.tif
   Output filename: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250112day.tif
   [MEMORY] Initial: 427.5 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memo

   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: float32
   [NODATA] Using nodata value -9999 for float32 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250112day.tif
   [MEMORY] Final: 428.1 MB (Change: +0.7 MB)
   ✅ Generated and saved COG: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250112day.tif

[4/4] Processing: drcs_activations/202501_Fire_CA/aria/dist/dist_hls/OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250114.tif
   Output filename: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250114day.tif
   [MEMORY] Initial: 428.1 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   

   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250114day.tif
   [MEMORY] Final: 428.1 MB (Change: +0.0 MB)
   ✅ Generated and saved COG: 202501_Fire_CA_OPERA-DIST-ALERT-HLS-VEG-DIST-STATUS_20250114day.tif

✅ Batch processing complete: 4 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/aria/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/aria/files_converted.csv

📊 BATCH PROCESSING SUMMARY
Total files processed: 4
Successful: 4
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-02T20:36:39.929478


In [40]:
# Process cir files with chunked processing
if k3:
    print("\n" + "="*50)
    print("🌊 Processing Water Mask Files (Chunked)")
    print("="*50)
    # Initialize combined results DataFrame
    all_files_processed = pd.DataFrame()
    
    # Use the chunked conversion function instead of the original
    def chunked_converter(name, cog_filename, cog_data_bucket, cog_data_prefix, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, cog_filename, cog_data_bucket, cog_data_prefix, 
            local_output_dir, chunk_config=CHUNK_CONFIG
        )
    
    wm_results = process_file_batch(
        file_list=k3,
        s3_client=s3_client,
        config=cir_bucket,
        filename_creator_func=create_cog_filename_k3,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True
    )
    all_files_processed = pd.concat([all_files_processed, wm_results], ignore_index=True)
    
    # Print overall summary
    print_batch_summary(all_files_processed)


🌊 Processing Water Mask Files (Chunked)

[1/2] Processing: drcs_activations/202501_Fire_CA/aria/dist/dist_s1_prototype/disturbance_track64_2025-01-09_share.tif
   Output filename: 202501_Fire_CA_disturbance_track64_share_2025-01-09day.tif
   [MEMORY] Initial: 352.7 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB
   [CHUNKS] Processing 6 chunks (3x2)
   [BAND 1/1] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_disturbance_track64_share_2025-01-09day.tif
   [MEMORY] Final: 358.2 MB (Change: +5.5 MB)
   ✅ Generated and saved COG: 202501_Fire_CA_disturbance_track64_share_2025-01-09day.tif

[2/2] Processing: drcs_activations/202501_Fire_CA/aria/dist/dist_s1_prototype/disturbance_track71_2025-01-09_share.tif
   Output filename: 202501_Fire_CA_disturbance_track71_share_2025-01-09day.tif
   [MEMORY] Initial: 358.2 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:4326 using chunked processing...
📊 Optimal chunk size: 1024x1024
   Estimated memory per chunk: 1.00 MB

   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/HLS/aria/202501_Fire_CA_disturbance_track71_share_2025-01-09day.tif
   [MEMORY] Final: 358.2 MB (Change: +0.0 MB)
   ✅ Generated and saved COG: 202501_Fire_CA_disturbance_track71_share_2025-01-09day.tif

✅ Batch processing complete: 2 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/HLS/aria/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/HLS/aria/files_converted.csv

📊 BATCH PROCESSING SUMMARY
Total files processed: 2
Successful: 2
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-02T20:36:58.711421


In [ ]:
# Display final results (for a single instance)
print(f"\n📊 Final Processing Results:")
print(f"Total files processed: {len(all_files_processed)}")
print(f"\nProcessed files DataFrame:")
all_files_processed

## Memory Usage Summary

You can check the final memory usage and cleanup

In [41]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")


📊 Memory Usage Summary:
  Current memory usage: 358.2 MB
  Available memory: 28677.6 MB
  Memory percent used: 9.3%


## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.